# **Guardrails in CrewAI**

## This is an **Enterprise feature of CrewAI**
## You need to **signup** for an Enterprise account to use this feature.


## Install necessary Libraries

In [ ]:
!pip install -q crewai crewai_tools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.8/195.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 836.4/836.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.0/

In [ ]:
import crewai
import crewai_tools
print(crewai.__version__)
print(crewai_tools.__version__)

1.15.10
1.15.10


# Set API Keys

In [ ]:
from google.colab import userdata
import os
os.environ["SERPER_API_KEY"] = userdata.get('SERPER_API_KEY')
os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENROUTER_API_KEY')

## Import Dependencies

In [ ]:
from crewai import Agent, Task, Crew, LLM

## Create the LLM Object

In [ ]:
# OPENROUTER hosted LLMs
llm = LLM(
    model="openrouter/openai/gpt-oss-120b",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

guardrail_llm = llm

# Define Agents

In [ ]:
from crewai.tasks.hallucination_guardrail import HallucinationGuardrail

# Guardrail with explicit reference context
guardrail = HallucinationGuardrail(
    context="AI helps with various tasks including analysis and generation.",
    llm=guardrail_llm,
)

# Custom threshold validation
# Strict guardrail requiring high faithfulness score
strict_guardrail = HallucinationGuardrail(
    context="Quantum computing uses qubits that exist in superposition states.",
    llm=guardrail_llm,
    threshold=8.0  # Requires score >= 8 to pass validation
)

# Guardrail with tool response context
weather_guardrail = HallucinationGuardrail(
    context="Current weather information for the requested location",
    llm=guardrail_llm,
    tool_response="Weather API returned: Temperature 28°C, Humidity 65%, Clear skies"
)




[2026-08-25 06:25:53][WARNING]: Hallucination detection is a no-op in open source, use it for free at https://app.crewai.com


[2026-08-25 06:25:53][WARNING]: Hallucination detection is a no-op in open source, use it for free at https://app.crewai.com


[2026-08-25 06:25:53][WARNING]: Hallucination detection is a no-op in open source, use it for free at https://app.crewai.com



# Define the Crew with Agents and Tasks

In [ ]:
from crewai import Agent, Task, Crew
from crewai_tools import SerperDevTool

search_tool = SerperDevTool()

# Add tools to agent
assistant = Agent(
    role="AI Agent",
    goal="""Helpful assistant who answeres user query: {query}""",
    backstory="""You are an experienced and helpful assistant who responds to user's query: {query}.""",
    llm=llm,
    tools=[search_tool],
    verbose=True
)

# Create your task with the guardrail
task = Task(
    description="Answer the user query {query}",
    expected_output="A brief reply to the query: {query}.",
    agent=assistant,
    guardrail=weather_guardrail  # Add the guardrail to validate output
)

crew = Crew(
  agents=[assistant],
  tasks=[task],
  verbose=True,
  cache=True
)

## Execute the Crew

In [ ]:
results = await crew.kickoff_async(
                                   inputs={"query": "How is the weather in Gurugram now?"}
    )

print(results)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 471f21da-8713-41f4-b46d-bee324b7a426                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Answer the user query How is the weather in Gurugram now?                                                │
│  ID: 9afc8a2d-5b4f-44d7-a82b-3751d5d12bd8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Agent                                                                                                │
│                                                                                                                 │
│  Task: Answer the user query How is the weather in Gurugram now?                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#4) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_the_internet_with_serper                                                                          │
│  Args: {'search_query': 'current weather Gurugram'}                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool search_the_internet_with_serper executed with result: {'searchParameters': {'q': 'current weather Gurugram', 'type': 'search', 'num': 10, 'engine': 'google'}, 'organic': [{'title': 'Gurgaon, Haryana, India Weather Forecast', 'link': 'https://www.accuweat...


...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Answer the user query How is the weather in Gurugram now?                                                │
│  Agent: AI Agent                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── 🛡️ Guardrail Success ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Guardrail Passed                                                                                               │
│  Name: Validation Successful                                                                                    │
│  Status: ✅ Validated                                                                                           │
│  Attempts: 1                                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

The latest reports show that Gurugram is currently **around 81 °F (≈ 27 °C)** with **partly sunny / cloudy** skies. The RealFeel temperature is about **93 °F (≈ 34 °C)**. Humidity is roughly **70 %–75 %**, and winds are light (around **5–7 km/h**). There is no heavy rain at the moment.


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 471f21da-8713-41f4-b46d-bee324b7a426                                                                       │
│  Final Output: The latest reports show that Gurugram is currently **around 81 °F (≈ 27 °C)** with **partly      │
│  sunny / cloudy** skies. The RealFeel temperature is about **93 °F (≈ 34 °C)**. Humidity is roughly **70 %–75   │
│  %**, and winds are light (around **5–7 km/h**). There is no heavy rain at the moment.                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                                              │
│  To enable tracing later, do any one of these:                               │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                                                              │
╰─────────────────────────

# Define another Crew with Agents and Tasks

In [ ]:
from crewai import Agent, Task, Crew
from crewai_tools import SerperDevTool

search_tool = SerperDevTool()

# Add tools to agent
ai_assistant = Agent(
    role="AI Agent",
    goal="""Helpful assistant who answeres user query: {query}""",
    backstory="""You are an experienced and helpful assistant who responds to user's query: {query}.""",
    llm=llm,
    tools=[search_tool],
    verbose=True
)

# Create your task with the guardrail
task = Task(
    description="Answer the user query {query}",
    expected_output="A brief reply to the query: {query}.",
    agent=ai_assistant,
    guardrail=strict_guardrail  # Add the guardrail to validate output
)

crew = Crew(
  agents=[assistant],
  tasks=[task],
  verbose=True,
  cache=True
)

## Run the Crew

In [ ]:
results = await crew.kickoff_async(
                                   inputs={"query": "What is superposition in Quantum computing?"}
    )

print(results)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 842fe14d-6587-4272-98f3-5009de8f7408                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Answer the user query What is superposition in Quantum computing?                                        │
│  ID: 5ccf95c2-3485-464f-a696-559d05f1152b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Agent                                                                                                │
│                                                                                                                 │
│  Task: Answer the user query What is superposition in Quantum computing?                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Agent                                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Superposition is a fundamental principle of quantum mechanics that allows a quantum bit (qubit) to exist in a  │
│  combination of both the `|0⟩` and `|1⟩` states at the same time. Instead of being limited to a single binary   │
│  value like a classical bit, a qubit can be described by the wave‑function                                      │
│                                                                                                                 │
│  \[                                                                                                             │
│  |\psi\rangle = \alpha|0\rangle + \beta|1\rangle,                                                               │
│  \]                                                                                                             │
│                                                                                                                 │
│  where the complex amplitudes α and β determine the probabilities of measuring the qubit as 0 or 1 (with        │
│  \(|\alpha|^2 + |\beta|^2 = 1\)). This simultaneous “being in both states” enables quantum computers to         │
│  process many possible outcomes in parallel, providing the basis for their potential speed‑up over classical    │
│  computers.                                                                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 🛡️ Guardrail Check ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Guardrail Evaluation Started                                                                                   │
│  Name: HallucinationGuardrail (no-op)                                                                           │
│  Status: 🔄 Evaluating                                                                                          │
│  Attempt: 1                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Answer the user query What is superposition in Quantum computing?                                        │
│  Agent: AI Agent                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── 🛡️ Guardrail Success ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Guardrail Passed                                                                                               │
│  Name: Validation Successful                                                                                    │
│  Status: ✅ Validated                                                                                           │
│  Attempts: 1                                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Superposition is a fundamental principle of quantum mechanics that allows a quantum bit (qubit) to exist in a combination of both the `|0⟩` and `|1⟩` states at the same time. Instead of being limited to a single binary value like a classical bit, a qubit can be described by the wave‑function  

\[
|\psi\rangle = \alpha|0\rangle + \beta|1\rangle,
\]

where the complex amplitudes α and β determine the probabilities of measuring the qubit as 0 or 1 (with \(|\alpha|^2 + |\beta|^2 = 1\)). This simultaneous “being in both states” enables quantum computers to process many possible outcomes in parallel, providing the basis for their potential speed‑up over classical computers.


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 842fe14d-6587-4272-98f3-5009de8f7408                                                                       │
│  Final Output: Superposition is a fundamental principle of quantum mechanics that allows a quantum bit (qubit)  │
│  to exist in a combination of both the `|0⟩` and `|1⟩` states at the same time. Instead of being limited to a   │
│  single binary value like a classical bit, a qubit can be described by the wave‑function                        │
│                                                                                                                 │
│  \[                                                                                                             │
│  |\psi\rangle = \alpha|0\rangle + \beta|1\rangle,                                                               │
│  \]                                                                                                             │
│                                                                                                                 │
│  where the complex amplitudes α and β determine the probabilities of measuring the qubit as 0 or 1 (with        │
│  \(|\alpha|^2 + |\beta|^2 = 1\)). This simultaneous “being in both states” enables quantum computers to         │
│  process many possible outcomes in parallel, providing the basis for their potential speed‑up over classical    │
│  computers.                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯